# Kirsch Visualizer — Near-Wellbore Stresses

Interactive-style demo of the **Kirsch solution** for stresses on the borehole wall.

Uses `NearWellboreStressesCalculation.calculate_kirsch_borehole_wall_stresses`
and `calculate_principal_stresses_analytical` from GeomechPy.

Polar plots show radial, tangential and axial stress components around the wellbore circumference.


## Setup & imports


In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "Project":
    REPO_ROOT = REPO_ROOT.parent.parent
elif REPO_ROOT.name == "example":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
# Interactive plots that render in JupyterLab, VS Code and nbviewer.
pio.renderers.default = "plotly_mimetype+notebook_connected"

from geomechpy.near_wellbore_stresses import NearWellboreStressesCalculation

print("Imports OK — repo root:", REPO_ROOT)

Imports OK — repo root: /home/user/GeomechPy_smolrun


## Input parameters (single depth)

Edit these values to explore different stress regimes and well trajectories.


In [2]:
# Far-field stresses (psi)
SHMIN = 6500.0
SHMAX = 8500.0
SVERT = 10000.0
PORE_PRESSURE = 4500.0
MUD_PRESSURE = 5000.0

# Orientation
SHMAX_AZIMUTH = 30.0       # deg from geographic North
BOREHOLE_DEVIATION = 0.0   # 0 = vertical well
BOREHOLE_AZIMUTH = 0.0     # deg

# Rock property
POISSON_RATIO_STATIC = 0.25

# Circumferential sampling (deg relative to Top-of-Hole)
theta = np.linspace(0, 360, 361)

print(f"Stress regime check: Sv={SVERT}, SHmax={SHMAX}, Shmin={SHMIN}")


Stress regime check: Sv=10000.0, SHmax=8500.0, Shmin=6500.0


## Compute Kirsch borehole-wall stresses


In [3]:
wall = NearWellboreStressesCalculation.calculate_kirsch_borehole_wall_stresses(
    shmin=SHMIN,
    shmax=SHMAX,
    svert=SVERT,
    pore_pressure=PORE_PRESSURE,
    shmax_azimuth=SHMAX_AZIMUTH,
    mud_pressure=MUD_PRESSURE,
    theta=theta,
    poisson_ratio_static=POISSON_RATIO_STATIC,
    borehole_deviation=BOREHOLE_DEVIATION,
    borehole_azimuth=BOREHOLE_AZIMUTH,
)

principals = NearWellboreStressesCalculation.calculate_principal_stresses_analytical(
    sigma_tt=wall.sigma_tt,
    sigma_zz=wall.sigma_zz,
    sigma_tz=wall.sigma_tz,
)

print("sigma_rr range (psi):", wall.sigma_rr.min(), "–", wall.sigma_rr.max())
print("sigma_tt range (psi):", wall.sigma_tt.min(), "–", wall.sigma_tt.max())
print("sigma_zz range (psi):", wall.sigma_zz.min(), "–", wall.sigma_zz.max())
print("sigma_1  range (psi):", principals.sigma_1.min(), "–", principals.sigma_1.max())


sigma_rr range (psi): 500.0 – 500.0
sigma_tt range (psi): 10499.999999999998 – 18500.0
sigma_zz range (psi): 9000.0 – 11000.0
sigma_1  range (psi): 10500.0 – 18500.0


## Polar plot — stress components around the borehole


In [4]:
# Interactive polar plots (0deg = Top-of-Hole, clockwise around the wellbore)
fig = make_subplots(
    rows=1, cols=3,
    specs=[[{"type": "polar"}, {"type": "polar"}, {"type": "polar"}]],
    subplot_titles=("σθθ (ksi)", "σzz (ksi)",
                    "σ₁ principal (ksi)"),
)


def add_polar(col, r, line_color, fill_color):
    fig.add_trace(
        go.Scatterpolar(r=r, theta=theta, mode="lines",
                        line=dict(color=line_color, width=2),
                        fill="toself", fillcolor=fill_color,
                        showlegend=False),
        row=1, col=col,
    )


add_polar(1, wall.sigma_tt / 1000, "blue", "rgba(0,0,255,0.20)")
add_polar(2, wall.sigma_zz / 1000, "green", "rgba(0,128,0,0.20)")
add_polar(3, principals.sigma_1 / 1000, "red", "rgba(255,0,0,0.20)")

for i in range(1, 4):
    key = "polar" if i == 1 else f"polar{i}"
    fig.update_layout(**{key: dict(
        angularaxis=dict(direction="clockwise", rotation=90),
    )})

fig.update_layout(
    height=460, width=1200, template="plotly_white",
    title=(f"Kirsch wall stresses | SHmax az={SHMAX_AZIMUTH}° | "
           f"dev={BOREHOLE_DEVIATION}° az={BOREHOLE_AZIMUTH}° | "
           f"Pw={MUD_PRESSURE:.0f} psi"),
    margin=dict(t=90),
)
fig.show()

## Cartesian view — stress vs azimuth


In [5]:
theta_deg = theta  # deg from Top-of-Hole

fig = go.Figure()
fig.add_trace(go.Scatter(x=theta_deg, y=wall.sigma_tt / 1000, mode="lines",
                         name="σθθ", line=dict(color="blue")))
fig.add_trace(go.Scatter(x=theta_deg, y=wall.sigma_zz / 1000, mode="lines",
                         name="σzz", line=dict(color="green")))
fig.add_trace(go.Scatter(x=theta_deg, y=wall.sigma_rr / 1000, mode="lines",
                         name="σrr", line=dict(color="black", dash="dash")))
fig.add_trace(go.Scatter(x=theta_deg, y=principals.sigma_1 / 1000, mode="lines",
                         name="σ₁", line=dict(color="red", width=2)))
fig.add_trace(go.Scatter(x=theta_deg, y=principals.sigma_2 / 1000, mode="lines",
                         name="σ₂", line=dict(color="magenta", width=2)))
fig.add_hline(y=0, line=dict(color="gray", width=1))
fig.update_layout(
    title="Borehole-wall stresses vs azimuth",
    xaxis=dict(title="θ (deg from Top-of-Hole)", range=[0, 360]),
    yaxis_title="Stress (ksi)", template="plotly_white",
    height=460, width=950,
)
fig.show()

## Compare vertical vs horizontal well

Same far-field stresses; only borehole deviation changes.


In [6]:
def run_kirsch(deviation):
    w = NearWellboreStressesCalculation.calculate_kirsch_borehole_wall_stresses(
        shmin=SHMIN, shmax=SHMAX, svert=SVERT,
        pore_pressure=PORE_PRESSURE, shmax_azimuth=SHMAX_AZIMUTH,
        mud_pressure=MUD_PRESSURE, theta=theta,
        poisson_ratio_static=POISSON_RATIO_STATIC,
        borehole_deviation=deviation, borehole_azimuth=BOREHOLE_AZIMUTH,
    )
    p = NearWellboreStressesCalculation.calculate_principal_stresses_analytical(
        sigma_tt=w.sigma_tt, sigma_zz=w.sigma_zz, sigma_tz=w.sigma_tz,
    )
    return w, p


wall_v, prin_v = run_kirsch(0.0)
wall_h, prin_h = run_kirsch(90.0)

fig = make_subplots(
    rows=1, cols=2, specs=[[{"type": "polar"}, {"type": "polar"}]],
    subplot_titles=("Vertical well (dev=0°)", "Horizontal well (dev=90°)"),
)


def add_pair(col, prin):
    show = col == 1
    fig.add_trace(go.Scatterpolar(r=prin.sigma_1 / 1000, theta=theta, mode="lines",
                                  line=dict(color="red", width=2),
                                  name="σ₁", showlegend=show), row=1, col=col)
    fig.add_trace(go.Scatterpolar(r=prin.sigma_2 / 1000, theta=theta, mode="lines",
                                  line=dict(color="blue", width=2),
                                  name="σ₂", showlegend=show), row=1, col=col)


add_pair(1, prin_v)
add_pair(2, prin_h)

for i in (1, 2):
    key = "polar" if i == 1 else f"polar{i}"
    fig.update_layout(**{key: dict(
        angularaxis=dict(direction="clockwise", rotation=90),
    )})

fig.update_layout(height=500, width=1000, template="plotly_white",
                  title="Principal wall stresses — vertical vs horizontal",
                  margin=dict(t=90))
fig.show()

## Notes

- **$\theta = 0°$** is Top-of-Hole (TOH).
- For a vertical well, breakouts form where $\\sigma_{\theta\theta}$ is maximum (typically along Shmin direction).
- Tensile fractures initiate where $\\sigma_{\theta\theta}$ is minimum (along SHmax).
- Change `BOREHOLE_DEVIATION`, `SHMAX_AZIMUTH` or mud pressure above and re-run to explore the stress state.
